Air Traffic Control

Chromosome - list of the rules like the hight, distance, altitude of each plane flying in the air avoiding crash.

## Basic Imports

In [28]:
import numpy as np
import pandas as pd
from copy import deepcopy

In [29]:
def closest_pass(p1, p2, v1, v2, time_window, minimum_distance):
    delta_p = p1 - p2
    delta_v = v1 - v2
    a = np.dot(delta_v, delta_v) # Speed difference between the planes
    b = 2 * np.dot(delta_p, delta_v)# Relative speed between the planes
    c = np.dot(delta_p, delta_p) # Distance between the planes

    if a == 0:
        return np.sqrt(c)
    t = -b / (2 * a)

    if t < 0:
        return 0
    if t > time_window:
        return 0    
        
    closest_distance = np.sqrt(c + t * (b + a * t))
    
    if closest_distance < minimum_distance:
        return -1000
    return 0


def convert_long_lat_alt_to_cartesian(long, lat, alt):
    y=lat * 111111
    x = long * 111111 * np.cos(np.radians(lat))
    z = alt
    return np.array([x, y, z])


def get_cartesian_velocity(vertrate, speed, heading):
    heading_rad = np.radians(heading)

    v_z = speed * np.cos(heading_rad)
    v_x = speed * np.sin(heading_rad)
    v_y = vertrate 
    
    return np.array([v_x, v_y, v_z])

## Test

In [30]:
#plane 1
lat1 =49.5354448739
long1 =2.83373033678
speed1 =181.779353382
head_1 =66.4820383263
vertrate1 = 4.55168
alt1 = 4838.7

#plane 2
lat2 =51.7063293457
long2 =-3.28788138725
speed2 = 236.560348391
head_2 =316.762391024
vertrate2 = 0.32512
alt2 = 11841.48

#plane 3
lat3 =49.542388916
long3 =2.85810771741
speed3 = 181.985246682
head_3 =246.3335256222
vertrate3 = 5.20192
alt3 = 4892.04

In [31]:
pos1 = convert_long_lat_alt_to_cartesian(long1, lat1, alt1)
vel1 = get_cartesian_velocity(vertrate1, speed1, head_1)

pos2 = convert_long_lat_alt_to_cartesian(long2, lat2, alt2)
vel2 = get_cartesian_velocity(vertrate2, speed2, head_2) 

pos3 = convert_long_lat_alt_to_cartesian(long3, lat3, alt3)
vel3 = get_cartesian_velocity(vertrate3, speed3, head_3)

In [32]:
print("Plane 1 position:", pos1, "velocity:", vel1)
print("Plane 3 position:", pos3, "velocity:", vel3)

closest_pass(pos1, pos3, vel1, vel3, 60, 6000)

Plane 1 position: [2.04336159e+05 5.50393282e+06 4.83870000e+03] velocity: [166.679856   4.55168   72.536604]
Plane 3 position: [2.06064693e+05 5.50470437e+06 4.89204000e+03] velocity: [-166.679856    5.20192   -73.051048]


-1000

# Import Dataset

In [33]:
flights_df = pd.read_csv("Data/new_flights.csv")
print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548410  4ca1d2  49.535445  2.833730  181.779353   66.482038   4.55168   
1  1527548410  4ca5ea  51.706329 -3.287881  236.560348  316.762391   0.32512   
2  1527548410  4ca8d7  53.679060 -5.311567  162.050677  269.818109  -4.87680   
3  1527548410  4ca303  53.778488 -6.063354  123.908406  261.163409  -2.60096   
4  1527548410  4ca97c  50.252563 -3.978424  226.215014  338.101807   0.00000   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0  RYR52GH      False  False  False   674.0       4701.54      4838.70   
1  RYR17XT      False  False  False  5701.0      11582.40     11841.48   
2  RYR73UF      False  False  False  2522.0       3817.62      3992.88   
3  RYR39BN      False  False  False  2002.0       2141.22      2270.76   
4  RYR74UE      False  False  False  7474.0      11582.40     11833.86   

   lastposupdate   lastcontact  
0   1.527548e+09  1.527548e+09  
1   1.52

In [34]:
positions = []
velocities = []

for _, row in flights_df.iterrows():
    pos = convert_long_lat_alt_to_cartesian(row["lon"], row["lat"], row["geoaltitude"])
    vel = get_cartesian_velocity(row["vertrate"], row["velocity"], row["heading"])
    positions.append(pos)
    velocities.append(vel)

flights_df["converted_position"] = positions
flights_df["cartesian_velocity"] = velocities

print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548410  4ca1d2  49.535445  2.833730  181.779353   66.482038   4.55168   
1  1527548410  4ca5ea  51.706329 -3.287881  236.560348  316.762391   0.32512   
2  1527548410  4ca8d7  53.679060 -5.311567  162.050677  269.818109  -4.87680   
3  1527548410  4ca303  53.778488 -6.063354  123.908406  261.163409  -2.60096   
4  1527548410  4ca97c  50.252563 -3.978424  226.215014  338.101807   0.00000   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0  RYR52GH      False  False  False   674.0       4701.54      4838.70   
1  RYR17XT      False  False  False  5701.0      11582.40     11841.48   
2  RYR73UF      False  False  False  2522.0       3817.62      3992.88   
3  RYR39BN      False  False  False  2002.0       2141.22      2270.76   
4  RYR74UE      False  False  False  7474.0      11582.40     11833.86   

   lastposupdate   lastcontact  \
0   1.527548e+09  1.527548e+09   
1   1.

In [35]:
TIME_WINDOW = 100      
MIN_DISTANCE = 6000   

conflict_pairs = []
n = len(flights_df)

for i in range(n):
    for j in range(i + 1, n):
        p1 = flights_df.iloc[i]["converted_position"]
        p2 = flights_df.iloc[j]["converted_position"]
        v1 = flights_df.iloc[i]["cartesian_velocity"]
        v2 = flights_df.iloc[j]["cartesian_velocity"]

        score = closest_pass(p1, p2, v1, v2, TIME_WINDOW, MIN_DISTANCE)

        if score < 0:
            conflict_pairs.append({
                "plane_A": flights_df.iloc[i]["callsign"].strip(),
                "plane_B": flights_df.iloc[j]["callsign"].strip(),
                "conflict_score": score
            })

conflicts_df = pd.DataFrame(conflict_pairs)

if conflicts_df.empty:
    print("No conflicts detected within the time window.")
else:
    print(f"Conflicts: {len(conflicts_df)}")
    print(conflicts_df.to_string(index=False))


Conflicts: 142
plane_A plane_B  conflict_score
RYR74UE RYR74UE           -1000
RYR66CU RYR66CU           -1000
RYR66CU RYR66CU           -1000
 EIN529  EIN529           -1000
EIN76HJ EIN76HJ           -1000
RYR74UE RYR74UE           -1000
RYR74UE RYR74UE           -1000
 YR52GH    52GH           -1000
RYR87FC RYR87FC           -1000
 EIN529  EIN529           -1000
 RY72GH    52GH           -1000
 RY72GH   RY2GH           -1000
RYR66CU RYR66CU           -1000
EIN76HJ EIN76HJ           -1000
EIN76HJ EIN76HJ           -1000
EIN76HJ EIN76HJ           -1000
 EIN7RE  EIN7RE           -1000
 EIN7RE  EIN7RE           -1000
   RYR5    52GH           -1000
   RYR5   RY2GH           -1000
   RYR5 RYR8976           -1000
   52GH   RY2GH           -1000
   52GH RYR8976           -1000
   52GH R0052GH           -1000
RYR66CU RYR66CU           -1000
RYR66CU RYR66CU           -1000
EIN76HJ EIN76HJ           -1000
EIN76HJ EIN76HJ           -1000
EIN76HJ EIN76HJ           -1000
EIN76HJ EIN76HJ          

#### Find all planes' distance to the Dublins' runways

In [36]:
runways_df = pd.read_csv("Data/dublin_runways.csv")
print(runways_df.head())

       id  airport_ref airport_ident  length_ft  width_ft surface  lighted  \
0  356512         2533          EIDW    10203.0     148.0     CON        1   
1  235634         2533          EIDW     8652.0     148.0    ASPH        1   
2  235636         2533          EIDW     6798.0     148.0     ASP        1   

   closed le_ident  le_latitude_deg  le_longitude_deg  le_elevation_ft  \
0       0      10L         53.43716          -6.28062            235.0   
1       0      10R         53.42243          -6.29007            242.0   
2       0       16         53.43700          -6.26198            217.0   

   le_heading_degT  le_displaced_threshold_ft he_ident  he_latitude_deg  \
0             97.0                        NaN      28R        53.435200   
1             95.0                        NaN      28L        53.420300   
2            157.0                        NaN       34        53.419899   

   he_longitude_deg  he_elevation_ft  he_heading_degT  \
0          -6.24496            2

#### Convert runway threshold positions to Cartesian

In [37]:
runway_thresholds = []

for _, row in runways_df.iterrows():
    low_end_pos = convert_long_lat_alt_to_cartesian(row["le_longitude_deg"], row["le_latitude_deg"], 0)
    high_end_pos = convert_long_lat_alt_to_cartesian(row["he_longitude_deg"], row["he_latitude_deg"], 0)
    runway_thresholds.append({
        "runway_name": str(row["le_ident"]) + "/" + str(row["he_ident"]),
        "le_ident": row["le_ident"],
        "he_ident": row["he_ident"],
        "le_pos": low_end_pos,
        "he_pos": high_end_pos,
        "le_heading": row["le_heading_degT"],
        "he_heading": row["he_heading_degT"],
        "length_ft": row["length_ft"]
    })

print(runway_thresholds)

[{'runway_name': '10L/28R', 'le_ident': '10L', 'he_ident': '28R', 'le_pos': array([-415709.68442539, 5937456.28476   ,       0.        ]), 'he_pos': array([-413368.4400408, 5937238.5072   ,       0.       ]), 'le_heading': 97.0, 'he_heading': 278.0, 'length_ft': 10203.0}, {'runway_name': '10R/28L', 'le_ident': '10R', 'he_ident': '28L', 'le_pos': array([-416479.47665226, 5935819.61973   ,       0.        ]), 'he_pos': array([-413885.48966363, 5935582.9533    ,       0.        ]), 'le_heading': 95.0, 'he_heading': 275.0, 'length_ft': 8652.0}, {'runway_name': '16/34', 'le_ident': '16', 'he_ident': '34', 'le_pos': array([-414477.47693878, 5937438.507     ,       0.        ]), 'he_pos': array([-413823.83894118, 5935538.397789  ,       0.        ]), 'le_heading': 157.0, 'he_heading': 337.0, 'length_ft': 6798.0}]


#### Distance to Runway and Landing Readiness Check

ICAO Standard: 3 degree glide path

Final approach intercept: between 3 NM and 10 NM from threshold

Rule of thumb for Top of Descent (TOD): 3 nautical miles per 1,000 feet of altitude

1 nautical mile = 1852 meters

Calculate the horizontal distance (in meters) from a plane to a runway threshold.

In [53]:
NM_TO_METERS = 1852
MAX_APPROACH_DISTANCE_NM = 10       # 10 nautical miles from threshold
MAX_APPROACH_ALTITUDE_FT = 3000     # 3000 ft above runway elevation
DUBLIN_AIRPORT_ELEVATION_FT = 242

MAX_APPROACH_DISTANCE_M = MAX_APPROACH_DISTANCE_NM * NM_TO_METERS  # 18,520 m
MAX_APPROACH_ALT_TOTAL = DUBLIN_AIRPORT_ELEVATION_FT + MAX_APPROACH_ALTITUDE_FT  # 3242 ft

def get_distance_to_runway(plane_pos, runway_pos):
    dx = plane_pos[0] - runway_pos[0]
    dy = plane_pos[1] - runway_pos[1]
    
    horizontal_distance = np.sqrt(dx * dx + dy * dy)
    
    return horizontal_distance

Calculate the distance at which a plane should begin its approach.

ICAO standard: 3 NM per 1,000 feet of altitude (3 degree glide path).

In [54]:
def get_landing_start_distance(plane_alt_ft):
    altitude_thousands = plane_alt_ft / 1000.0
    distance_nm = altitude_thousands * 3.0
    distance_meters = distance_nm * NM_TO_METERS
    
    return distance_meters

Find the closest runway to a plane, considering:
1. Horizontal distance to each runway threshold
2. Which end of the runway to approach based on plane heading

Returns the closest runway info and whether the plane is within landing range.

In [56]:
MAX_APPROACH_DISTANCE_NM = 10       
MAX_APPROACH_ALTITUDE_FT = 3000 

def find_closest_runway(plane_lat, plane_lon, plane_alt, plane_heading):

    plane_pos = convert_long_lat_alt_to_cartesian(plane_lon, plane_lat, plane_alt)
    
    best_runway_name = ""
    best_approach_end = ""
    best_distance = 0
    best_heading_diff = 0

    for rw in runway_thresholds:
        dist_low_end = get_distance_to_runway(plane_pos, rw["le_pos"])
        dist_high_end = get_distance_to_runway(plane_pos, rw["he_pos"])

        # Check heading alignment with lown end approach
        # When landing on low end, plane flies heading = le_heading
        diff_le = abs(plane_heading - rw["le_heading"])
        if diff_le > 180:
            diff_le = 360 - diff_le

        # Check heading alignment with high end approach
        # When landing on high end, plane flies heading = he_heading
        diff_he = abs(plane_heading - rw["he_heading"])
        if diff_he > 180:
            diff_he = 360 - diff_he

        # Pick the end that best matches the plane heading
        if diff_le < diff_he:
            approach_dist = dist_low_end
            approach_end = rw["le_ident"]
            heading_diff = diff_le
        else:
            approach_dist = dist_high_end
            approach_end = rw["he_ident"]
            heading_diff = diff_he

        # Pick the runway with the shortest distance
        if approach_dist < best_distance:
            best_distance = approach_dist
            best_runway_name = rw["runway_name"]
            best_approach_end = approach_end
            best_heading_diff = heading_diff


        distance_ok = False
        altitude_ok = False
 
        if best_distance <= MAX_APPROACH_DISTANCE_M:
            distance_ok = True
    
        if plane_alt <= MAX_APPROACH_ALT_TOTAL:
            altitude_ok = True

        # Check if plane is within landing approach range
        landing_start_dist = get_landing_start_distance(plane_alt)

        is_ready = False
        status = ""

        if distance_ok and altitude_ok:
            is_ready = True
            status = "READY TO LAND"
        elif distance_ok and not altitude_ok:
            is_ready = False
            status = "TOO HIGH"
        elif not distance_ok and altitude_ok:
            is_ready = False
            status = "TOO FAR"
        else:
            is_ready = False
            status = "TOO FAR & TOO HIGH"

        return best_runway_name, best_approach_end, best_distance, best_heading_diff, distance_ok, altitude_ok, is_ready, status

Check all planes

In [58]:
ready_count = 0
too_high_count = 0
too_far_count = 0
too_far_high_count = 0

for idx, row in flights_df.iterrows():
    callsign = row["callsign"].strip()
    lat = row["lat"]
    lon = row["lon"]
    alt = row["geoaltitude"]
    heading = row["heading"]
    
    rw_name, appr_end, dist, hdg_diff, dist_ok, alt_ok, is_ready, status = find_closest_runway(lat, lon, alt, heading)
    
    dist_km = dist / 1000.0

    if is_ready:
        ready_count = ready_count + 1
    elif status == "TOO HIGH":
        too_high_count = too_high_count + 1
    elif status == "TOO FAR":
        too_far_count = too_far_count + 1
    else:
        too_far_high_count = too_far_high_count + 1
 
total = ready_count + too_high_count + too_far_count + too_far_high_count

print(f"READY TO LAND:      {ready_count}")
print(f"TOO HIGH:           {too_high_count}")
print(f"TOO FAR:            {too_far_count}")
print(f"TOO FAR & TOO HIGH: {too_far_high_count}")
print(f"TOTAL PLANES:       {total}")

READY TO LAND:      64
TOO HIGH:           435
TOO FAR:            0
TOO FAR & TOO HIGH: 0
TOTAL PLANES:       499
